# Nueva sección

In [8]:
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')


#---------------------CONFIGURACION GENERAL---------------------------

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_coldwidth', None)
pd.set_option('display.width', None)

#---------------------VARIABLES---------------------------------------

mis_estados = [['Puebla', 6583278],['San Luis Potosí', 2822255],['Tabasco', 2402598]]
select_edo = mis_estados[0]

meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

primavera   = ['Marzo','Abril','Mayo']
verano      = ['Junio','Julio','Agosto']
otoño       = ['Septiembre','Octubre','Noviembre']
invierno    = ['Diciembre','Enero','Febrero']

trimestre1 = ['Enero','Febrero','Marzo']
trimestre2 = ['Abril','Mayo','Junio']
trimestre3 = ['Julio','Agosto','Septiembre']
trimestre4 = ['Octubre','Noviembre','Diciembre']

principales_delitos = ['Amenazas','Daño a la propiedad','Fraude','Lesiones','Otros delitos del Fuero Común','Robo','Violencia familiar']

#---------------------DATAFRAMES-------------------------------------

df_2015_2025 = pd.read_csv('/content/drive/MyDrive/Municipal-Delitos - (2015-2025) dic25_corte_sep26.csv', encoding='latin1')
df_2026 = pd.read_csv('/content/drive/MyDrive/RNID-Delitos_Municipal-2026-sep2026.csv', encoding='utf-8-sig')

#--------------------DATAFRAME FINAL---------------------------------

df = pd.concat([df_2015_2025,df_2026], ignore_index=True)
df['Total_Anual'] = df[meses].sum(axis=1)
total_nacional = df.groupby(['Año'])['Total_Anual'].sum().reset_index()

df = df[df['Entidad'].isin(['Puebla','San Luis Potosí','Tabasco'])][['Año','Entidad','Municipio','Bien jurídico afectado', 'Tipo de delito','Subtipo de delito','Modalidad','Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']]

#--------------------VARIABLES PARA MIS CONSULTAS----------------------

estado = df[df['Entidad']==select_edo[0]]
municipios = estado['Municipio'].unique()
bja = estado['Bien jurídico afectado'].unique()
td = estado['Tipo de delito'].unique()
std = estado['Subtipo de delito'].unique()
md = estado['Modalidad'].unique()
estado['Total_Anual'] = estado[meses].sum(axis=1)
dt = estado.groupby('Año')['Total_Anual'].sum().reset_index()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_2027/3692058868.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  estado['Total_Anual'] = estado[meses].sum(axis=1)


In [9]:
meses_transcurridos = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre']
x_valores = np.arange(1, 10)
x_futuros = np.array([10, 11, 12])

def proyectar_meses(fila):
    y_valores = fila[meses_transcurridos].values.astype(float)
    m, b = np.polyfit(x_valores, y_valores, 1)
    predicciones = (m * x_futuros) + b
    predicciones_limpias = np.maximum(predicciones, 0)
    return pd.Series(np.round(predicciones_limpias).astype(int), index=['Octubre', 'Noviembre', 'Diciembre'])

estado.loc[df['Año']==2026,['Octubre','Noviembre','Diciembre']] = estado[estado['Año']==2026].apply(proyectar_meses, axis=1)

In [21]:
print(f"Indice de delitos Anual en {select_edo[0]} 2015-2026")
indice_delictivo = estado.groupby(['Año'])['Total_Anual'].sum().reset_index()
indice_delictivo['ind_anual'] = (indice_delictivo['Total_Anual']/total_nacional['Total_Anual'])*100
indice_delictivo

Indice de delitos Anual en Puebla 2015-2026


,Año,Total_Anual,ind_anual
0,2015,64399,3.884597
1,2016,51061,2.894321
2,2017,53800,2.773918
3,2018,61172,3.074080
4,2019,76556,3.696249
5,2020,63587,3.453580
6,2021,75141,3.675716
7,2022,77106,3.599737
8,2023,77244,3.553852
9,2024,80005,3.823983


In [22]:
print(f"Los 3 años mas peligrosos en {select_edo[0]}")
primeros_tres = indice_delictivo.sort_values(by='Total_Anual', ascending=False).head(3)
primeros_tres

Los 3 años mas peligrosos en Puebla


,Año,Total_Anual,ind_anual
10,2025,82122,4.072033
9,2024,80005,3.823983
8,2023,77244,3.553852


In [23]:
print(f"Indice de delitos Anual en {select_edo} 2015-2025")
ind_delict_poblac = estado.groupby(['Año'])['Total_Anual'].sum().reset_index()
ind_delict_poblac['id_poblacional'] = (ind_delict_poblac['Total_Anual']/6583278)
ind_delict_poblac

Indice de delitos Anual en ['Puebla', 6583278] 2015-2025


,Año,Total_Anual,id_poblacional
0,2015,64399,0.009782
1,2016,51061,0.007756
2,2017,53800,0.008172
3,2018,61172,0.009292
4,2019,76556,0.011629
5,2020,63587,0.009659
6,2021,75141,0.011414
7,2022,77106,0.011712
8,2023,77244,0.011733
9,2024,80005,0.012153


In [25]:
delitos_promedio = estado.groupby(['Año','Tipo de delito'])['Total_Anual'].sum().reset_index()
delitos_promedio['Porcentaje'] = (delitos_promedio['Total_Anual']/delitos_promedio['Año'].map(dt.set_index('Año')['Total_Anual']))*100
datos_promedio = delitos_promedio[['Año','Tipo de delito','Porcentaje']].sort_values(by=['Año','Porcentaje'], ascending=False).groupby('Año').head(1)
print("Los delitos con mayor contribucion por año en los primeros 5 años\n")
datos_promedio.head(5)


Los delitos con mayor contribucion por año en los primeros 5 años



,Año,Tipo de delito,Porcentaje
477,2026,Robo,36.841349
432,2025,Robo,39.889433
392,2024,Robo,40.427473
352,2023,Robo,41.198022
312,2022,Robo,39.113688


In [29]:
print("Los delitos con mayor contribucion por año en los ultimos 5 años\n")
datos_promedio.head(-7)

Los delitos con mayor contribucion por año en los ultimos 5 años



,Año,Tipo de delito,Porcentaje
477,2026,Robo,36.841349
432,2025,Robo,39.889433
392,2024,Robo,40.427473
352,2023,Robo,41.198022
312,2022,Robo,39.113688


In [20]:
trimestre_mayor = estado[['Año','Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre',]]
trimestre_mayor['t1'] = trimestre_mayor[trimestre1].sum(axis=1)
trimestre_mayor['t2'] = trimestre_mayor[trimestre2].sum(axis=1)
trimestre_mayor['t3'] = trimestre_mayor[trimestre3].sum(axis=1)
trimestre_mayor['t4'] = trimestre_mayor[trimestre4].sum(axis=1)
trimestre_mayor = trimestre_mayor[['Año','t1','t2','t3','t4']].groupby(by='Año')[['t1','t2','t3','t4']].sum().reset_index()
trimestre_mayor['Trimestre_Mayor'] = trimestre_mayor[['t1','t2','t3','t4']].idxmax(axis=1)
print(f"Trimestre con mayor numero de delitos por año en {select_edo[0]}\n")
trimestre_mayor.head(15)

Trimestre con mayor numero de delitos por año en Puebla



/tmp/ipykernel_2027/910410897.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trimestre_mayor['t1'] = trimestre_mayor[trimestre1].sum(axis=1)
/tmp/ipykernel_2027/910410897.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trimestre_mayor['t2'] = trimestre_mayor[trimestre2].sum(axis=1)
/tmp/ipykernel_2027/910410897.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentatio

,Año,t1,t2,t3,t4,Trimestre_Mayor
0,2015,15768,16874,16243,15514,t2
1,2016,14837,16296,9752,10176,t2
2,2017,12357,13722,13836,13885,t4
3,2018,14279,15477,15477,15939,t4
4,2019,18126,21746,19820,16864,t2
5,2020,16064,14053,16078,17392,t4
6,2021,17380,19505,18696,19560,t4
7,2022,18995,20427,19161,18523,t2
8,2023,18876,20340,19304,18724,t2
9,2024,18809,20348,21083,19765,t3


In [30]:
delitos_habitantes = estado[estado['Tipo de delito'].isin(principales_delitos)]
delitos_habitantes = delitos_habitantes[['Tipo de delito','Total_Anual']]
delitos_habitantes = delitos_habitantes.groupby(by='Tipo de delito')['Total_Anual'].sum().reset_index()
delitos_habitantes['Porcentaje'] = delitos_habitantes['Total_Anual']/select_edo[1]
delitos_habitantes = delitos_habitantes.sort_values('Total_Anual', ascending=False)
print(f"Los 7 delitos principales realacionados con la poblacion de {select_edo[0]}\n")
delitos_habitantes

Los 7 delitos principales realacionados con la poblacion de Puebla



,Tipo de delito,Total_Anual,Porcentaje
5,Robo,347092,0.052723
6,Violencia familiar,97913,0.014873
3,Lesiones,68070,0.010340
0,Amenazas,49457,0.007513
2,Fraude,45914,0.006974
1,Daño a la propiedad,38325,0.005822
4,Otros delitos del Fuero Común,32678,0.004964


In [31]:
promedio_estacion = estado[meses]
total_delitos = dt['Total_Anual'].sum()
promedio_estacion['Primavera'] = promedio_estacion[primavera].sum(axis=1)
promedio_estacion['Verano'] = promedio_estacion[verano].sum(axis=1)
promedio_estacion['Otoño'] = promedio_estacion[otoño].sum(axis=1)
promedio_estacion['Invierno'] = promedio_estacion[invierno].sum(axis=1)
promedio_estacion = promedio_estacion[['Primavera','Verano','Otoño','Invierno']].sum().to_frame().T
promedio_estacion['Primavera'] = (promedio_estacion['Primavera']/total_delitos)*100
promedio_estacion['Verano'] = (promedio_estacion['Verano']/total_delitos)*100
promedio_estacion['Otoño'] = (promedio_estacion['Otoño']/total_delitos)*100
promedio_estacion['Invierno'] = (promedio_estacion['Invierno']/total_delitos)*100
print(f"Promedio de delitos por estacion del año en {select_edo[0]}\n")
promedio_estacion

Promedio de delitos por estacion del año en Puebla



/tmp/ipykernel_2027/2683442138.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  promedio_estacion['Primavera'] = promedio_estacion[primavera].sum(axis=1)
/tmp/ipykernel_2027/2683442138.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  promedio_estacion['Verano'] = promedio_estacion[verano].sum(axis=1)
/tmp/ipykernel_2027/2683442138.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in

,Primavera,Verano,Otoño,Invierno
0,27.036516,26.070268,25.731172,23.915547


In [34]:
año_max = dt[dt['Total_Anual']==dt['Total_Anual'].max()]
año_min = dt[dt['Total_Anual']==dt['Total_Anual'].min()]
print("Año con mayor cantidad de delitos")
año_max

Año con mayor cantidad de delitos


,Año,Total_Anual
10,2025,82122


In [35]:
print("Año con menor cantidad de delitos")
año_min

Año con menor cantidad de delitos


,Año,Total_Anual
1,2016,51061


In [43]:
print(f"PREDICCIÓN PARA EL RESTO DEL AÑO 2026 EN {select_edo[0]}\n")
estado[meses].sum().to_frame().T

PREDICCIÓN PARA EL RESTO DEL AÑO 2026 EN Puebla



,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,65282,65445,75572,72011,75425,73036,72112,69890,69171,73834,69236,66538


In [46]:
print(f"PREDICCION LOGISTICA (HOMBRE/MUJER)")
clasificacion = estado.groupby(['Año','Tipo de delito','Subtipo de delito'])['Total_Anual'].sum().reset_index()
clasificacion['Hombres'] = (clasificacion['Total_Anual']*0.947).round().astype(int)
clasificacion['Mujeres'] = (clasificacion['Total_Anual']*0.053).round().astype(int)
clasificacion.head(10)

PREDICCION LOGISTICA (HOMBRE/MUJER)


,Año,Tipo de delito,Subtipo de delito,Total_Anual,Hombres,Mujeres
0,2015,Aborto,Aborto,7,7,0
1,2015,Abuso de confianza,Abuso de confianza,1271,1204,67
2,2015,Abuso sexual,Abuso sexual,0,0,0
3,2015,Acoso sexual,Acoso sexual,116,110,6
4,2015,Allanamiento de morada,Allanamiento de morada,370,350,20
5,2015,Amenazas,Amenazas,4401,4168,233
6,2015,Contra el medio ambiente,Contra el medio ambiente,20,19,1
7,2015,Corrupción de menores,Corrupción de menores,41,39,2
8,2015,Daño a la propiedad,Daño a la propiedad,4366,4135,231
9,2015,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,42,40,2
